## Exploratory Data Analysis and Cleanup for Children's Books

In [1]:
# importing libraries
import pandas as pd
pd.set_option("display.max_columns", None)
import os
from dotenv import load_dotenv
import numpy as np
from langdetect import detect, DetectorFactory
from tqdm.auto import tqdm
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
import ast
from collections import OrderedDict
import re

/Users/navyajain/Downloads/build/agentic-book-recommendation/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DetectorFactory.seed = 42

In [3]:
# paths
load_dotenv()
#children_path = os.getenv("CHILDREN_BOOKS")

True

In [4]:
#print(children_path)

In [5]:
comic_df = pd.read_json('../data/books/goodreads_books_poetry.json', lines=True)

In [6]:
comic_df.shape

(36514, 29)

In [7]:
comic_df.head()

,isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
0,,1,[],US,eng,"[{'count': '8', 'name': 'to-read'}, {'count': ...",,false,3.83,,[],Number 30 in a series of literary pamphlets pu...,Paperback,https://www.goodreads.com/book/show/16037549-v...,"[{'author_id': '15585', 'role': ''}]","Houghton, Mifflin and Company",80,1,,11,,1887,https://www.goodreads.com/book/show/16037549-v...,https://images.gr-assets.com/books/1348176637m...,16037549,3,5212748,Vision of Sir Launfal and Other Poems,Vision of Sir Launfal and Other Poems
1,0811223981,2,[],US,,"[{'count': '100', 'name': 'to-read'}, {'count'...",,false,3.83,B00U2WY9U8,[],Fairy Tales gathers the unconventional verse d...,Paperback,https://www.goodreads.com/book/show/22466716-f...,"[{'author_id': '16073', 'role': ''}, {'author_...",New Directions,128,20,9780811223980,4,,2015,https://www.goodreads.com/book/show/22466716-f...,https://images.gr-assets.com/books/1404958407m...,22466716,37,41905435,Fairy Tales: Dramolettes,Fairy Tales: Dramolettes
2,0374428115,7,[],US,,"[{'count': '32', 'name': 'to-read'}, {'count':...",,false,4.38,,[],Three poems describe the nighttime adventures ...,Paperback,https://www.goodreads.com/book/show/926662.Gro...,"[{'author_id': '18540', 'role': ''}, {'author_...",Farrar Straus Giroux,,12,9780374428112,7,,2008,https://www.goodreads.com/book/show/926662.Gro...,https://s.gr-assets.com/assets/nophoto/book/11...,926662,45,911665,Growltiger's Last Stand and Other Poems,Growltiger's Last Stand and Other Poems
3,0156182890,12,[],US,,"[{'count': '554', 'name': 'to-read'}, {'count'...",,false,3.71,B00IWTRB1W,"[1230072, 315167, 676169, 18522, 124335, 88263...",A modern verse play about the search for meani...,Paperback,https://www.goodreads.com/book/show/926667.The...,"[{'author_id': '18540', 'role': ''}]",Mariner Books,190,18,9780156182898,3,,1964,https://www.goodreads.com/book/show/926667.The...,https://images.gr-assets.com/books/1382939971m...,926667,115,995066,The Cocktail Party,The Cocktail Party
4,1942004192,4,[],US,eng,"[{'count': '228', 'name': 'to-read'}, {'count'...",,false,5.00,,"[25869488, 23630890, 25448131, 25464039, 42166...",Louder Than Everything You Love is about trans...,Paperback,https://www.goodreads.com/book/show/29065952-l...,"[{'author_id': '14308759', 'role': ''}]",ELJ Publications,118,23,9781942004196,12,First,2015,https://www.goodreads.com/book/show/29065952-l...,https://images.gr-assets.com/books/1455198396m...,29065952,9,49294781,Louder Than Everything You Love,Louder Than Everything You Love


In [8]:
df = comic_df.copy()

| Column | Meaning |
|---|---|
| `isbn` | 10-digit ISBN for the book edition, if available. |
| `text_reviews_count` | Number of written/text reviews for this book edition. |
| `series` | List of Goodreads series IDs the book belongs to. Empty list means no series or unavailable. |
| `country_code` | Country code from Goodreads metadata, often `US`. |
| `language_code` | Language code for the edition, such as `eng`, `en-US`, etc. Can be blank. |
| `popular_shelves` | User-created Goodreads shelf tags for the book, with shelf name and count. |
| `asin` | Amazon Standard Identification Number, if available. |
| `is_ebook` | Whether the edition is an ebook, stored as `"true"` or `"false"`. |
| `average_rating` | Average Goodreads rating for the book, usually from 1 to 5. |
| `kindle_asin` | Kindle-specific Amazon ID, if available. |
| `similar_books` | List of Goodreads `book_id`s marked as similar books. |
| `description` | Book description or blurb text. Can be empty. |
| `format` | Book format, such as `Hardcover`, `Paperback`, or `Kindle Edition`. |
| `link` | Goodreads link for the book page. |
| `authors` | List of author objects, usually containing `author_id` and `role`. |
| `publisher` | Publisher name for this edition. |
| `num_pages` | Number of pages, if available. |
| `publication_day` | Publication day of the month, if available. |
| `isbn13` | 13-digit ISBN for the book edition, if available. |
| `publication_month` | Publication month, if available. |
| `edition_information` | Extra edition information, such as `Anniversary Edition` or `First Edition`. |
| `publication_year` | Publication year for this edition. |
| `url` | Goodreads URL for the book page. Usually similar to `link`. |
| `image_url` | URL for the book cover image. |
| `book_id` | Goodreads ID for this specific book edition. Useful for joining with reviews/interactions. |
| `ratings_count` | Number of ratings for this book edition. |
| `work_id` | Goodreads work ID. Groups multiple editions of the same underlying book. Useful for deduplication. |
| `title` | Full book title as shown on Goodreads, often including series information. |
| `title_without_series` | Book title with series information removed. Better for clean display/search. |

In [9]:
df.shape

(36514, 29)

In [10]:
df.columns

Index(['isbn', 'text_reviews_count', 'series', 'country_code', 'language_code',
       'popular_shelves', 'asin', 'is_ebook', 'average_rating', 'kindle_asin',
       'similar_books', 'description', 'format', 'link', 'authors',
       'publisher', 'num_pages', 'publication_day', 'isbn13',
       'publication_month', 'edition_information', 'publication_year', 'url',
       'image_url', 'book_id', 'ratings_count', 'work_id', 'title',
       'title_without_series'],
      dtype='str')

In [11]:
df = df[['title', 'title_without_series', 'series', 'country_code', 'language_code',
       'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id', 'book_id',
       'text_reviews_count', 'average_rating', 'ratings_count', 'popular_shelves',
       'is_ebook', 'format', 'num_pages', 'publisher', 'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [12]:
df.shape

(36514, 26)

In [13]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,Vision of Sir Launfal and Other Poems,Vision of Sir Launfal and Other Poems,[],US,eng,,,,,5212748,16037549,1,3.83,3,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,Paperback,80,"Houghton, Mifflin and Company",1,11,1887,,Number 30 in a series of literary pamphlets pu...,"[{'author_id': '15585', 'role': ''}]",[]
1,Fairy Tales: Dramolettes,Fairy Tales: Dramolettes,[],US,,0811223981,,B00U2WY9U8,9780811223980,41905435,22466716,2,3.83,37,"[{'count': '100', 'name': 'to-read'}, {'count'...",false,Paperback,128,New Directions,20,4,2015,,Fairy Tales gathers the unconventional verse d...,"[{'author_id': '16073', 'role': ''}, {'author_...",[]
2,Growltiger's Last Stand and Other Poems,Growltiger's Last Stand and Other Poems,[],US,,0374428115,,,9780374428112,911665,926662,7,4.38,45,"[{'count': '32', 'name': 'to-read'}, {'count':...",false,Paperback,,Farrar Straus Giroux,12,7,2008,,Three poems describe the nighttime adventures ...,"[{'author_id': '18540', 'role': ''}, {'author_...",[]
3,The Cocktail Party,The Cocktail Party,[],US,,0156182890,,B00IWTRB1W,9780156182898,995066,926667,12,3.71,115,"[{'count': '554', 'name': 'to-read'}, {'count'...",false,Paperback,190,Mariner Books,18,3,1964,,A modern verse play about the search for meani...,"[{'author_id': '18540', 'role': ''}]","[1230072, 315167, 676169, 18522, 124335, 88263..."
4,Louder Than Everything You Love,Louder Than Everything You Love,[],US,eng,1942004192,,,9781942004196,49294781,29065952,4,5.00,9,"[{'count': '228', 'name': 'to-read'}, {'count'...",false,Paperback,118,ELJ Publications,23,12,2015,First,Louder Than Everything You Love is about trans...,"[{'author_id': '14308759', 'role': ''}]","[25869488, 23630890, 25448131, 25464039, 42166..."


In [14]:
df['clean_title'] = df['title_without_series']

In [15]:
df.shape

(36514, 27)

### Detecting languages

In [16]:
def detect_lang(text):
    if pd.isna(text) or len(str(text).strip()) < 30:
        return np.nan

    try:
        lang = detect(str(text))
        return "eng" if lang == "en" else lang
    except:
        return np.nan

In [17]:
# normalize empty strings to NaN
df["language_code_final"] = df["language_code"].replace("", np.nan)

# create title + description text
df["language_text"] = (
    df["clean_title"].fillna("") + " " + df["description"].fillna("")
)

# detect only missing language codes
missing_lang_mask = df["language_code_final"].isna()

texts_to_detect = df.loc[missing_lang_mask, "language_text"]

detected_languages = [
    detect_lang(text)
    for text in tqdm(texts_to_detect, desc="Detecting languages")
]

df.loc[missing_lang_mask, "detected_language_code"] = detected_languages

# fill missing language code with detected language
df["language_code_final"] = df["language_code_final"].fillna(
    df["detected_language_code"]
)

# normalize English variants
df["language_code_final"] = (
    df["language_code_final"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "en": "eng",
        "en-us": "eng",
        "en-gb": "eng",
        "en-ca": "eng"
    })
)

Detecting languages: 100%|██████████| 19462/19462 [00:40<00:00, 478.64it/s]


In [18]:
df.head()

,title,title_without_series,series,country_code,language_code,isbn,asin,kindle_asin,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,is_ebook,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,clean_title,language_code_final,language_text,detected_language_code
0,Vision of Sir Launfal and Other Poems,Vision of Sir Launfal and Other Poems,[],US,eng,,,,,5212748,16037549,1,3.83,3,"[{'count': '8', 'name': 'to-read'}, {'count': ...",false,Paperback,80,"Houghton, Mifflin and Company",1,11,1887,,Number 30 in a series of literary pamphlets pu...,"[{'author_id': '15585', 'role': ''}]",[],Vision of Sir Launfal and Other Poems,eng,Vision of Sir Launfal and Other Poems Number 3...,nan
1,Fairy Tales: Dramolettes,Fairy Tales: Dramolettes,[],US,,0811223981,,B00U2WY9U8,9780811223980,41905435,22466716,2,3.83,37,"[{'count': '100', 'name': 'to-read'}, {'count'...",false,Paperback,128,New Directions,20,4,2015,,Fairy Tales gathers the unconventional verse d...,"[{'author_id': '16073', 'role': ''}, {'author_...",[],Fairy Tales: Dramolettes,eng,Fairy Tales: Dramolettes Fairy Tales gathers t...,eng
2,Growltiger's Last Stand and Other Poems,Growltiger's Last Stand and Other Poems,[],US,,0374428115,,,9780374428112,911665,926662,7,4.38,45,"[{'count': '32', 'name': 'to-read'}, {'count':...",false,Paperback,,Farrar Straus Giroux,12,7,2008,,Three poems describe the nighttime adventures ...,"[{'author_id': '18540', 'role': ''}, {'author_...",[],Growltiger's Last Stand and Other Poems,eng,Growltiger's Last Stand and Other Poems Three ...,eng
3,The Cocktail Party,The Cocktail Party,[],US,,0156182890,,B00IWTRB1W,9780156182898,995066,926667,12,3.71,115,"[{'count': '554', 'name': 'to-read'}, {'count'...",false,Paperback,190,Mariner Books,18,3,1964,,A modern verse play about the search for meani...,"[{'author_id': '18540', 'role': ''}]","[1230072, 315167, 676169, 18522, 124335, 88263...",The Cocktail Party,eng,The Cocktail Party A modern verse play about t...,eng
4,Louder Than Everything You Love,Louder Than Everything You Love,[],US,eng,1942004192,,,9781942004196,49294781,29065952,4,5.00,9,"[{'count': '228', 'name': 'to-read'}, {'count'...",false,Paperback,118,ELJ Publications,23,12,2015,First,Louder Than Everything You Love is about trans...,"[{'author_id': '14308759', 'role': ''}]","[25869488, 23630890, 25448131, 25464039, 42166...",Louder Than Everything You Love,eng,Louder Than Everything You Love Louder Than Ev...,nan


In [19]:
df.columns

Index(['title', 'title_without_series', 'series', 'country_code',
       'language_code', 'isbn', 'asin', 'kindle_asin', 'isbn13', 'work_id',
       'book_id', 'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'is_ebook', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'clean_title', 'language_code_final', 'language_text',
       'detected_language_code'],
      dtype='str')

In [20]:
df.shape

(36514, 30)

In [21]:
df=df[['clean_title', 'series', 'language_code_final', 
       'isbn', 'isbn13', 'work_id', 'book_id', 
       'text_reviews_count', 'average_rating', 'ratings_count',
       'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books']]

In [22]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books
0,Vision of Sir Launfal and Other Poems,[],eng,,,5212748,16037549,1,3.83,3,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Paperback,80,"Houghton, Mifflin and Company",1,11,1887,,Number 30 in a series of literary pamphlets pu...,"[{'author_id': '15585', 'role': ''}]",[]
1,Fairy Tales: Dramolettes,[],eng,0811223981,9780811223980,41905435,22466716,2,3.83,37,"[{'count': '100', 'name': 'to-read'}, {'count'...",Paperback,128,New Directions,20,4,2015,,Fairy Tales gathers the unconventional verse d...,"[{'author_id': '16073', 'role': ''}, {'author_...",[]
2,Growltiger's Last Stand and Other Poems,[],eng,0374428115,9780374428112,911665,926662,7,4.38,45,"[{'count': '32', 'name': 'to-read'}, {'count':...",Paperback,,Farrar Straus Giroux,12,7,2008,,Three poems describe the nighttime adventures ...,"[{'author_id': '18540', 'role': ''}, {'author_...",[]
3,The Cocktail Party,[],eng,0156182890,9780156182898,995066,926667,12,3.71,115,"[{'count': '554', 'name': 'to-read'}, {'count'...",Paperback,190,Mariner Books,18,3,1964,,A modern verse play about the search for meani...,"[{'author_id': '18540', 'role': ''}]","[1230072, 315167, 676169, 18522, 124335, 88263..."
4,Louder Than Everything You Love,[],eng,1942004192,9781942004196,49294781,29065952,4,5.00,9,"[{'count': '228', 'name': 'to-read'}, {'count'...",Paperback,118,ELJ Publications,23,12,2015,First,Louder Than Everything You Love is about trans...,"[{'author_id': '14308759', 'role': ''}]","[25869488, 23630890, 25448131, 25464039, 42166..."


In [23]:
df.shape

(36514, 21)

In [24]:
df = df[
    df["language_code_final"].astype("string").str.strip().str.lower() == "eng"
].copy()

In [25]:
df.shape

(25429, 21)

In [26]:
df['format'].value_counts().shape

(134,)

In [27]:
unique_formats = df["format"].dropna().unique().tolist()
print(unique_formats)

['Paperback', 'Hardcover', 'ebook', '', 'chapbook', 'Kindle Edition', 'Audio CD', 'paperback', 'Unknown Binding', 'Leather Bound', 'Audio', 'paper', 'poetry chapbook', 'Mass Market Paperback', 'Chapbook', 'Board book', 'newsprint stapled folio', 'html', 'Poetry', 'cloth', 'Casebound', 'Audible Audio', 'Audiobook', 'Board Book', 'Verse in categories', 'hardcover', 'hardback', 'Letterpressed Chapbook', 'chapbook/ ebook', 'iPad', 'E-chap', 'audio cd', 'Hardbound', 'chap book/e-book', 'Zine', 'ebook / paperback', 'MP3 Book', 'Library Binding', 'Hardback', 'Paperback, e-book', 'loose pages in an embossed box', 'Twelve Audio Cassettes', 'Paper and Photographic Cardboard', 'saddle-stitched chapbook', 'Poetry Chapbook', 'Handmade Chapbook', 'Micro-Chapbook', 'Pamphlet', 'Hand-Crafted Chapbook', 'zine', 'hand-sewn chapbook', 'ribbon-bound chapbook', 'Comic', 'Hand-sewn Binding', 'School &amp; Library Binding', 'cards in envelope', 'Audio Cassette', 'Other Format', 'poetry', 'e-chapbook', 'Saddl

In [29]:
# 1. Define readable formats to keep
keep_formats = {
    # Standard print
    "Paperback",
    "paperback",
    "Hardcover",
    "hardcover",
    "Hardback",
    "hardback",
    "Hardbound",
    "Hard Cover",
    "Hardcover first edition",
    "Hardback, cloth",
    "Hardcover, Sewn Binding, Paper Dust Jacket",
    "Hardcover/Paperback",
    "Hardcover and Paperback",
    "HC",
    "pbk",
    "paper",
    "Trade Paperback",
    "Trade paperback",
    "Trade Paper",
    "Mass Market Paperback",
    "Perfect Paperback",
    "softcover",
    "Softcover",
    "softback",
    "Softback",
    "Library Binding",
    "School &amp; Library Binding",
    "Textbook Binding",
    "Turtleback",
    "Leather Bound",
    "Leatherbound, hard-cover, 5 volumes",
    "cloth",
    "Casebound",
    "Perfect Bound",
    "perfect bound",
    "paperback perfect bound",
    "Publisher's Binding",
    "Wrappers",

    # Chapbooks / poetry formats
    "chapbook",
    "Chapbook",
    "poetry chapbook",
    "Poetry Chapbook",
    "Poetry chapbook",
    "Letterpressed Chapbook",
    "Handmade Chapbook",
    "Handmade chapbook",
    "Hand-Crafted Chapbook",
    "Hand-bound",
    "Hand-sewn Binding",
    "hand-sewn chapbook",
    "hand-stitched chapbook",
    "Limited Edition Hand-Stitched Chapbook",
    "Micro-Chapbook",
    "saddle-stitched chapbook",
    "Saddle-stitched Chapbook",
    "Saddle stitched chapbook",
    "saddle stitched chapbook",
    "Saddle stapled chapbook",
    "staple-bound chapbook",
    "poetry chapbook - saddle stitched",
    "Chapbook (Machine Stitched)",
    "Chapbook/Pamphlet",
    "chapbooks",
    "chap",
    "chapbook/ ebook",
    "chapbook/ebook",
    "chapbook/e-book",
    "chap book/e-book",
    "chapbook/e-chap",
    "chap/e-chap",
    "e-chapbook",
    "E-chapbook",
    "E-chap",
    "limited edition fine art chapbook",
    "sew-bound",
    "ribbon-bound chapbook",

    # Pamphlet / zine / magazine / handmade print
    "Pamphlet",
    "Zine",
    "zine",
    "Magazine",
    "newsprint stapled folio",
    "Screen-printed cover; stapled",
    "saddle-stitched",
    "Saddle-stitched",
    "japanese stab bound",
    "Unbound",
    "loose pages in an embossed box",
    "Paper and Photographic Cardboard",
    "Paper, Color Photo and Black Twine",
    "paper with animal stamps and staple",
    "Paper, Fabric and Cloth",
    "cards in envelope",
    "Poem Cards",

    # Board / comic / artist book
    "Board book",
    "Board Book",
    "Comic",
    "Artist Book",
    "Hardcover Art and Poetry Gift Book",
    "wee book",

    # Ebooks / digital readable text
    "ebook",
    "Kindle Edition",
    "PDF",
    "PDF (Movie Script)",
    "html",
    "Digital publication",
    "print",
    "iPad",
    "Paperback, e-book",
    "Paperback, E-book",
    "Paperback, kindle &amp; ebook.",
    "Paperback, Kindle eBook",
    "ebook / paperback",

    # Foreign labels that are readable print
    "edicao brochada",
}

# 2. Define formats to drop
drop_formats = {
    # Missing / unknown
    "",
    "Unknown Binding",
    "unknown",
    "Other Format",

    # Audio
    "Audio",
    "audio cd",
    "Audio CD",
    "Audible Audio",
    "Audiobook",
    "Audiobook LP",
    "Audio Cassette",
    "audio cassette",
    "Twelve Audio Cassettes",
    "2 audio cassettes",
    "MP3 Book",
    "MP3 CD",

    # Software / compressed / non-readable file container
    "CD-ROM",
    "zip",

    # Not format / too vague as format
    "Poetry",
    "poetry",
    "Verse in categories",
}


# 4. Clean the format column
df["format_clean"] = (df["format"].fillna("").astype(str).str.strip())

# 5. Assign each row to a format group
df["format_group"] = np.select(
    [
        df["format_clean"].isin(keep_formats),
        df["format_clean"].isin(drop_formats)
    ],
    ["text", "other"], default="review")

# 6. Check any unclassified formats
df["format_group"].value_counts()

format_group
text      21314
other      4114
review        1
Name: count, dtype: int64

In [30]:
df = df[df["format_group"] == "text"].copy()

In [31]:
df["format_group"].value_counts()

format_group
text    21314
Name: count, dtype: int64

In [32]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format,num_pages,publisher,publication_day,publication_month,publication_year,edition_information,description,authors,similar_books,format_clean,format_group
0,Vision of Sir Launfal and Other Poems,[],eng,,,5212748,16037549,1,3.83,3,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Paperback,80,"Houghton, Mifflin and Company",1,11,1887,,Number 30 in a series of literary pamphlets pu...,"[{'author_id': '15585', 'role': ''}]",[],Paperback,text
1,Fairy Tales: Dramolettes,[],eng,0811223981,9780811223980,41905435,22466716,2,3.83,37,"[{'count': '100', 'name': 'to-read'}, {'count'...",Paperback,128,New Directions,20,4,2015,,Fairy Tales gathers the unconventional verse d...,"[{'author_id': '16073', 'role': ''}, {'author_...",[],Paperback,text
2,Growltiger's Last Stand and Other Poems,[],eng,0374428115,9780374428112,911665,926662,7,4.38,45,"[{'count': '32', 'name': 'to-read'}, {'count':...",Paperback,,Farrar Straus Giroux,12,7,2008,,Three poems describe the nighttime adventures ...,"[{'author_id': '18540', 'role': ''}, {'author_...",[],Paperback,text
3,The Cocktail Party,[],eng,0156182890,9780156182898,995066,926667,12,3.71,115,"[{'count': '554', 'name': 'to-read'}, {'count'...",Paperback,190,Mariner Books,18,3,1964,,A modern verse play about the search for meani...,"[{'author_id': '18540', 'role': ''}]","[1230072, 315167, 676169, 18522, 124335, 88263...",Paperback,text
4,Louder Than Everything You Love,[],eng,1942004192,9781942004196,49294781,29065952,4,5.00,9,"[{'count': '228', 'name': 'to-read'}, {'count'...",Paperback,118,ELJ Publications,23,12,2015,First,Louder Than Everything You Love is about trans...,"[{'author_id': '14308759', 'role': ''}]","[25869488, 23630890, 25448131, 25464039, 42166...",Paperback,text


In [33]:
df.shape

(21314, 23)

In [34]:
df.columns

Index(['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format', 'num_pages', 'publisher',
       'publication_day', 'publication_month', 'publication_year',
       'edition_information', 'description', 'authors', 'similar_books',
       'format_clean', 'format_group'],
      dtype='str')

In [35]:
df = df[['clean_title', 'series', 'language_code_final', 'isbn', 'isbn13',
       'work_id', 'book_id', 'text_reviews_count', 'average_rating',
       'ratings_count', 'popular_shelves', 'format_clean', 'num_pages', 'publisher',
       'publication_year', 'description', 'authors', 'similar_books']]

In [36]:
df.head()

,clean_title,series,language_code_final,isbn,isbn13,work_id,book_id,text_reviews_count,average_rating,ratings_count,popular_shelves,format_clean,num_pages,publisher,publication_year,description,authors,similar_books
0,Vision of Sir Launfal and Other Poems,[],eng,,,5212748,16037549,1,3.83,3,"[{'count': '8', 'name': 'to-read'}, {'count': ...",Paperback,80,"Houghton, Mifflin and Company",1887,Number 30 in a series of literary pamphlets pu...,"[{'author_id': '15585', 'role': ''}]",[]
1,Fairy Tales: Dramolettes,[],eng,0811223981,9780811223980,41905435,22466716,2,3.83,37,"[{'count': '100', 'name': 'to-read'}, {'count'...",Paperback,128,New Directions,2015,Fairy Tales gathers the unconventional verse d...,"[{'author_id': '16073', 'role': ''}, {'author_...",[]
2,Growltiger's Last Stand and Other Poems,[],eng,0374428115,9780374428112,911665,926662,7,4.38,45,"[{'count': '32', 'name': 'to-read'}, {'count':...",Paperback,,Farrar Straus Giroux,2008,Three poems describe the nighttime adventures ...,"[{'author_id': '18540', 'role': ''}, {'author_...",[]
3,The Cocktail Party,[],eng,0156182890,9780156182898,995066,926667,12,3.71,115,"[{'count': '554', 'name': 'to-read'}, {'count'...",Paperback,190,Mariner Books,1964,A modern verse play about the search for meani...,"[{'author_id': '18540', 'role': ''}]","[1230072, 315167, 676169, 18522, 124335, 88263..."
4,Louder Than Everything You Love,[],eng,1942004192,9781942004196,49294781,29065952,4,5.00,9,"[{'count': '228', 'name': 'to-read'}, {'count'...",Paperback,118,ELJ Publications,2015,Louder Than Everything You Love is about trans...,"[{'author_id': '14308759', 'role': ''}]","[25869488, 23630890, 25448131, 25464039, 42166..."


In [37]:
def clean_missing(x):
    if x is None:
        return np.nan
    if isinstance(x, str) and x.strip() == "":
        return np.nan
    if isinstance(x, list) and len(x) == 0:
        return np.nan
    return x


def unique_list(series):
    seen = set()
    result = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        key = str(x)

        if key not in seen:
            seen.add(key)
            result.append(x)

    return result


def first_non_missing(series):
    for x in series:
        x = clean_missing(x)

        if isinstance(x, float) and pd.isna(x):
            continue

        return x

    return np.nan


def longest_text(series):
    texts = []

    for x in series:
        x = clean_missing(x)

        if isinstance(x, str) and x.strip() != "":
            texts.append(x.strip())

    if len(texts) == 0:
        return np.nan

    return max(texts, key=len)


def max_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.max()


def mean_numeric(series):
    numeric = pd.to_numeric(series, errors="coerce")

    if numeric.notna().sum() == 0:
        return np.nan

    return numeric.mean()

In [38]:
numeric_cols = [
    "text_reviews_count",
    "average_rating",
    "ratings_count",
    "num_pages",
    "publication_year"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

In [39]:
agg_dict = {
    # Main identity fields
    "clean_title": first_non_missing,
    "authors": first_non_missing,
    "series": unique_list,
    "language_code_final": first_non_missing,

    # IDs for API calls / joins
    "isbn": unique_list,
    "isbn13": unique_list,
    "book_id": unique_list,

    # Edition-level metadata
    "publisher": unique_list,
    "publication_year": unique_list,

    # Book-level text/features
    "description": longest_text,
    "popular_shelves": first_non_missing,
    "similar_books": unique_list,

    # Numeric summary features
    "text_reviews_count": max_numeric,
    "ratings_count": max_numeric,
    "average_rating": mean_numeric,
    "num_pages": max_numeric,
}

# keep only columns that exist in df
agg_dict = {col: func for col, func in agg_dict.items() if col in df.columns}

books_work_df = (
    df.groupby("work_id", dropna=False)
    .agg(agg_dict)
    .reset_index()
)

In [40]:
rename_dict = {
    "series": "series_list",
    "isbn": "isbn_list",
    "isbn13": "isbn13_list",
    "book_id": "book_id_list",
    "publisher": "publisher_list",
    "publication_year": "publication_year_list",
    "similar_books": "similar_books_list",
}

books_work_df = books_work_df.rename(
    columns={k: v for k, v in rename_dict.items() if k in books_work_df.columns}
)

In [41]:
books_work_df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,166,Laocoon: An Essay on the Limits of Painting an...,"[{'author_id': '7723', 'role': ''}, {'author_i...",[],eng,[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],[1984.0],"Originally published in 1766, the Laocoonhas b...","[{'count': '76', 'name': 'to-read'}, {'count':...","[[814670, 30529, 117249, 2973210, 609042, 9490...",9,319,3.93,296.0
1,466,The Melancholy Death of Oyster Boy & Other Sto...,"[{'author_id': '5773', 'role': ''}]",[],eng,"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...","[2005.0, 2010.0, 1998.0, 1997.0, 2008.0]",Occupying a similarly sinister and macabre wor...,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[[205981, 193264, 36024, 47561, 223851, 58096,...",999,20556,4.13,128.0
2,527,Azul,"[{'author_id': '25575', 'role': ''}]",[],eng,"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","[2010.0, 2000.0]","La publicacion de 'Azul', primer libro de un e...","[{'count': '792', 'name': 'to-read'}, {'count'...","[[1073063, 167425, 1375835, 144611, 214319, 95...",1,10,3.88,140.0
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[{'author_id': '947', 'role': ''}, {'author_id...",[],eng,[0451530357],[9780451530356],[104870],[Signet],[2006.0],The plays collected here follow: the journeys ...,"[{'count': '41', 'name': 'to-read'}, {'count':...",[],4,31,3.97,736.0
4,1140,100 Selected Poems,"[{'author_id': '10547', 'role': ''}]",[],eng,[0802130720],[9780802130723],[76889],[Grove Press],[1994.0],E.E. Cummings is without question one of the m...,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[[75504, 367802, 65336, 47733, 133380, 46201, ...",369,22739,4.32,128.0


In [42]:
df = books_work_df.copy()

In [43]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,166,Laocoon: An Essay on the Limits of Painting an...,"[{'author_id': '7723', 'role': ''}, {'author_i...",[],eng,[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],[1984.0],"Originally published in 1766, the Laocoonhas b...","[{'count': '76', 'name': 'to-read'}, {'count':...","[[814670, 30529, 117249, 2973210, 609042, 9490...",9,319,3.93,296.0
1,466,The Melancholy Death of Oyster Boy & Other Sto...,"[{'author_id': '5773', 'role': ''}]",[],eng,"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...","[2005.0, 2010.0, 1998.0, 1997.0, 2008.0]",Occupying a similarly sinister and macabre wor...,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[[205981, 193264, 36024, 47561, 223851, 58096,...",999,20556,4.13,128.0
2,527,Azul,"[{'author_id': '25575', 'role': ''}]",[],eng,"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","[2010.0, 2000.0]","La publicacion de 'Azul', primer libro de un e...","[{'count': '792', 'name': 'to-read'}, {'count'...","[[1073063, 167425, 1375835, 144611, 214319, 95...",1,10,3.88,140.0
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[{'author_id': '947', 'role': ''}, {'author_id...",[],eng,[0451530357],[9780451530356],[104870],[Signet],[2006.0],The plays collected here follow: the journeys ...,"[{'count': '41', 'name': 'to-read'}, {'count':...",[],4,31,3.97,736.0
4,1140,100 Selected Poems,"[{'author_id': '10547', 'role': ''}]",[],eng,[0802130720],[9780802130723],[76889],[Grove Press],[1994.0],E.E. Cummings is without question one of the m...,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[[75504, 367802, 65336, 47733, 133380, 46201, ...",369,22739,4.32,128.0


In [44]:
df.shape

(16996, 17)

In [45]:
df.isna().sum()

work_id                     0
clean_title                 2
authors                     0
series_list                 0
language_code_final         0
isbn_list                   0
isbn13_list                 0
book_id_list                0
publisher_list              0
publication_year_list       0
description              1282
popular_shelves             0
similar_books_list          0
text_reviews_count          0
ratings_count               0
average_rating              0
num_pages                1066
dtype: int64

In [46]:
df.shape

(16996, 17)

In [47]:
df = df[df["description"].notna()].copy()

In [48]:
df.isna().sum()

work_id                    0
clean_title                1
authors                    0
series_list                0
language_code_final        0
isbn_list                  0
isbn13_list                0
book_id_list               0
publisher_list             0
publication_year_list      0
description                0
popular_shelves            0
similar_books_list         0
text_reviews_count         0
ratings_count              0
average_rating             0
num_pages                805
dtype: int64

In [49]:
df.shape

(15714, 17)

In [50]:
df.head()

,work_id,clean_title,authors,series_list,language_code_final,isbn_list,isbn13_list,book_id_list,publisher_list,publication_year_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,166,Laocoon: An Essay on the Limits of Painting an...,"[{'author_id': '7723', 'role': ''}, {'author_i...",[],eng,[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],[1984.0],"Originally published in 1766, the Laocoonhas b...","[{'count': '76', 'name': 'to-read'}, {'count':...","[[814670, 30529, 117249, 2973210, 609042, 9490...",9,319,3.93,296.0
1,466,The Melancholy Death of Oyster Boy & Other Sto...,"[{'author_id': '5773', 'role': ''}]",[],eng,"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...","[2005.0, 2010.0, 1998.0, 1997.0, 2008.0]",Occupying a similarly sinister and macabre wor...,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[[205981, 193264, 36024, 47561, 223851, 58096,...",999,20556,4.13,128.0
2,527,Azul,"[{'author_id': '25575', 'role': ''}]",[],eng,"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","[2010.0, 2000.0]","La publicacion de 'Azul', primer libro de un e...","[{'count': '792', 'name': 'to-read'}, {'count'...","[[1073063, 167425, 1375835, 144611, 214319, 95...",1,10,3.88,140.0
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[{'author_id': '947', 'role': ''}, {'author_id...",[],eng,[0451530357],[9780451530356],[104870],[Signet],[2006.0],The plays collected here follow: the journeys ...,"[{'count': '41', 'name': 'to-read'}, {'count':...",[],4,31,3.97,736.0
4,1140,100 Selected Poems,"[{'author_id': '10547', 'role': ''}]",[],eng,[0802130720],[9780802130723],[76889],[Grove Press],[1994.0],E.E. Cummings is without question one of the m...,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[[75504, 367802, 65336, 47733, 133380, 46201, ...",369,22739,4.32,128.0


In [51]:
df = df.drop(columns=["language_code_final", "publication_year_list"])

In [52]:
df.shape

(15714, 15)

In [53]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages
0,166,Laocoon: An Essay on the Limits of Painting an...,"[{'author_id': '7723', 'role': ''}, {'author_i...",[],[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],"Originally published in 1766, the Laocoonhas b...","[{'count': '76', 'name': 'to-read'}, {'count':...","[[814670, 30529, 117249, 2973210, 609042, 9490...",9,319,3.93,296.0
1,466,The Melancholy Death of Oyster Boy & Other Sto...,"[{'author_id': '5773', 'role': ''}]",[],"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...",Occupying a similarly sinister and macabre wor...,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[[205981, 193264, 36024, 47561, 223851, 58096,...",999,20556,4.13,128.0
2,527,Azul,"[{'author_id': '25575', 'role': ''}]",[],"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","La publicacion de 'Azul', primer libro de un e...","[{'count': '792', 'name': 'to-read'}, {'count'...","[[1073063, 167425, 1375835, 144611, 214319, 95...",1,10,3.88,140.0
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[{'author_id': '947', 'role': ''}, {'author_id...",[],[0451530357],[9780451530356],[104870],[Signet],The plays collected here follow: the journeys ...,"[{'count': '41', 'name': 'to-read'}, {'count':...",[],4,31,3.97,736.0
4,1140,100 Selected Poems,"[{'author_id': '10547', 'role': ''}]",[],[0802130720],[9780802130723],[76889],[Grove Press],E.E. Cummings is without question one of the m...,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[[75504, 367802, 65336, 47733, 133380, 46201, ...",369,22739,4.32,128.0


In [54]:
df["clean_title"] = (
    df["clean_title"]
    .astype(str)
    .str.replace(r"\s*\([^)]*#\d+[^)]*\)", "", regex=True)
    .str.strip()
)

In [55]:
def extract_author_ids(authors):
    if not isinstance(authors, list):
        return []

    return [
        str(a.get("author_id"))
        for a in authors
        if isinstance(a, dict) and a.get("author_id")
    ]

df["author_ids"] = df["authors"].apply(extract_author_ids)

In [56]:
def flatten_nested_list(x):
    if not isinstance(x, list):
        return []

    flat = []

    for item in x:
        if isinstance(item, list):
            flat.extend(item)
        else:
            flat.append(item)

    # remove missing/empty values and dedupe
    result = []
    seen = set()

    for item in flat:
        if item is None:
            continue

        item_str = str(item).strip()

        if item_str == "" or item_str.lower() in {"nan", "none"}:
            continue

        if item_str not in seen:
            seen.add(item_str)
            result.append(item)

    return result

In [57]:
for col in ["series_list", "similar_books_list"]:
    if col in df.columns:
        df[col] = df[col].apply(flatten_nested_list)

In [58]:
numeric_cols = [
    "text_reviews_count",
    "ratings_count",
    "average_rating",
    "num_pages"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [59]:
df["num_pages_missing"] = df["num_pages"].isna().astype(int)

In [60]:
for i, shelves in enumerate(df["popular_shelves"].head(10), start=1):
    print(f"\nRow {i}:")
    print(shelves)


Row 1:
[{'count': '76', 'name': 'to-read'}, {'count': '43', 'name': 'art'}, {'count': '26', 'name': 'philosophy'}, {'count': '20', 'name': 'currently-reading'}, {'count': '18', 'name': 'non-fiction'}, {'count': '14', 'name': 'poetry'}, {'count': '10', 'name': 'essays'}, {'count': '7', 'name': 'german'}, {'count': '6', 'name': 'german-literature'}, {'count': '6', 'name': 'aesthetics'}, {'count': '6', 'name': 'art-history'}, {'count': '5', 'name': 'owned'}, {'count': '5', 'name': 'classics'}, {'count': '5', 'name': '18th-century'}, {'count': '5', 'name': 'theory'}, {'count': '4', 'name': 'art-theory'}, {'count': '3', 'name': 'nonfiction'}, {'count': '3', 'name': 'literary-theory'}, {'count': '3', 'name': 'criticism'}, {'count': '3', 'name': 'lit-crit'}, {'count': '3', 'name': 'enlightenment'}, {'count': '2', 'name': 'history'}, {'count': '2', 'name': 'favorites'}, {'count': '2', 'name': 'arts'}, {'count': '2', 'name': 'unrated'}, {'count': '2', 'name': 'literature'}, {'count': '2', 'nam

In [61]:
int_df = df[['popular_shelves']]

In [62]:
int_df.head()

,popular_shelves
0,"[{'count': '76', 'name': 'to-read'}, {'count':..."
1,"[{'count': '12411', 'name': 'to-read'}, {'coun..."
2,"[{'count': '792', 'name': 'to-read'}, {'count'..."
3,"[{'count': '41', 'name': 'to-read'}, {'count':..."
4,"[{'count': '34800', 'name': 'to-read'}, {'coun..."


In [63]:
int_df.shape

(15714, 1)

In [64]:
int_df.to_csv('../data/intermediate/poetry_shelves.csv', index=False)

In [65]:
# TAG NORMALIZATION MAP ---> BUILT WITH CLAUDE
# instead of calling Claude or gpt's api or even using a model from HuggingFace, this seemed to be the fastest and the cheapest way
shelves = pd.read_csv('../data/intermediate/poetry_tag_mapping.csv')

In [66]:
shelves.head(20)

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,5195262,15551
1,poetry,poetry,keep,526110,15569
2,currently-reading,status,drop,170488,9831
3,classics,classics,keep,90746,2249
4,favorites,review,drop,74573,6129
5,fiction,fiction,keep,43404,3727
6,owned,ownership,drop,28943,4562
7,books-i-own,ownership,drop,24039,3682
8,non-fiction,nonfiction,keep,19479,3520
9,literature,classics,keep,17730,2602


In [67]:
vocab_df = shelves.copy()

In [68]:
tag_map = dict(
    zip(
        vocab_df.loc[vocab_df["action"] == "keep", "raw_tag"],
        vocab_df.loc[vocab_df["action"] == "keep", "clean_tag"]
    )
)

In [69]:
vocab_df.head()

,raw_tag,clean_tag,action,total_count,doc_freq
0,to-read,status,drop,5195262,15551
1,poetry,poetry,keep,526110,15569
2,currently-reading,status,drop,170488,9831
3,classics,classics,keep,90746,2249
4,favorites,review,drop,74573,6129


In [70]:
def replace_shelf_tags(shelves):
    # if shelves got saved as a string, convert it back to list
    if isinstance(shelves, str):
        try:
            shelves = ast.literal_eval(shelves)
        except Exception:
            return []

    if not isinstance(shelves, list):
        return []

    cleaned_shelves = []

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        raw_tag = shelf.get("name")

        # only replace if action == keep and raw_tag exists in tag_map
        if raw_tag in tag_map:
            new_shelf = shelf.copy()
            new_shelf["name"] = tag_map[raw_tag]
            cleaned_shelves.append(new_shelf)

    return cleaned_shelves

In [71]:
df["popular_shelves_cleaned"] = df["popular_shelves"].apply(replace_shelf_tags)

In [72]:
df[["popular_shelves", "popular_shelves_cleaned"]].head()

,popular_shelves,popular_shelves_cleaned
0,"[{'count': '76', 'name': 'to-read'}, {'count':...","[{'count': '43', 'name': 'art'}, {'count': '26..."
1,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[{'count': '862', 'name': 'poetry'}, {'count':..."
2,"[{'count': '792', 'name': 'to-read'}, {'count'...","[{'count': '36', 'name': 'poetry'}, {'count': ..."
3,"[{'count': '41', 'name': 'to-read'}, {'count':...","[{'count': '4', 'name': 'play'}, {'count': '4'..."
4,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[{'count': '1768', 'name': 'poetry'}, {'count'..."


In [73]:
print("Original popular_shelves:")
print(df.loc[df.index[0], "popular_shelves"])

print("\nCleaned popular_shelves:")
print(df.loc[df.index[0], "popular_shelves_cleaned"])

Original popular_shelves:
[{'count': '76', 'name': 'to-read'}, {'count': '43', 'name': 'art'}, {'count': '26', 'name': 'philosophy'}, {'count': '20', 'name': 'currently-reading'}, {'count': '18', 'name': 'non-fiction'}, {'count': '14', 'name': 'poetry'}, {'count': '10', 'name': 'essays'}, {'count': '7', 'name': 'german'}, {'count': '6', 'name': 'german-literature'}, {'count': '6', 'name': 'aesthetics'}, {'count': '6', 'name': 'art-history'}, {'count': '5', 'name': 'owned'}, {'count': '5', 'name': 'classics'}, {'count': '5', 'name': '18th-century'}, {'count': '5', 'name': 'theory'}, {'count': '4', 'name': 'art-theory'}, {'count': '3', 'name': 'nonfiction'}, {'count': '3', 'name': 'literary-theory'}, {'count': '3', 'name': 'criticism'}, {'count': '3', 'name': 'lit-crit'}, {'count': '3', 'name': 'enlightenment'}, {'count': '2', 'name': 'history'}, {'count': '2', 'name': 'favorites'}, {'count': '2', 'name': 'arts'}, {'count': '2', 'name': 'unrated'}, {'count': '2', 'name': 'literature'}, {

In [74]:
def merge_duplicate_shelf_tags(shelves):
    if not isinstance(shelves, list):
        return []

    merged = OrderedDict()

    for shelf in shelves:
        if not isinstance(shelf, dict):
            continue

        name = shelf.get("name")
        count = shelf.get("count", 0)

        if name is None or str(name).strip() == "":
            continue

        name = str(name).strip()

        try:
            count = int(count)
        except Exception:
            count = 0

        if name not in merged:
            merged[name] = count
        else:
            merged[name] += count

    return [
        {"count": str(count), "name": name}
        for name, count in merged.items()
    ]

In [75]:
df["popular_shelves_cleaned"] = df["popular_shelves_cleaned"].apply(
    merge_duplicate_shelf_tags
)

In [76]:
print(df.loc[df.index[1], "popular_shelves_cleaned"])

[{'count': '881', 'name': 'poetry'}, {'count': '289', 'name': 'fiction'}, {'count': '288', 'name': 'short-stories'}, {'count': '249', 'name': 'fantasy'}, {'count': '234', 'name': 'humor'}, {'count': '152', 'name': 'horror'}, {'count': '123', 'name': 'art'}, {'count': '89', 'name': 'graphic-novel'}, {'count': '126', 'name': 'children'}, {'count': '54', 'name': 'young-adult'}, {'count': '48', 'name': 'comics'}, {'count': '52', 'name': 'american-history'}, {'count': '18', 'name': 'picture-book'}, {'count': '15', 'name': '20th-century'}, {'count': '14', 'name': 'classics'}, {'count': '13', 'name': 'adult'}, {'count': '13', 'name': 'nonfiction'}, {'count': '12', 'name': 'contemporary'}, {'count': '10', 'name': 'fairy-tales'}, {'count': '7', 'name': 'paranormal'}]


In [77]:
for shelf in df.loc[df.index[10], "popular_shelves_cleaned"]:
    print(shelf)

{'count': '42', 'name': 'poetry'}
{'count': '13', 'name': 'horror'}
{'count': '12', 'name': 'fantasy'}
{'count': '9', 'name': 'fiction'}
{'count': '10', 'name': 'humor'}
{'count': '13', 'name': 'short-stories'}
{'count': '3', 'name': 'young-adult'}
{'count': '3', 'name': 'graphic-novel'}
{'count': '4', 'name': 'comics'}
{'count': '1', 'name': 'art'}
{'count': '3', 'name': 'children'}
{'count': '1', 'name': 'holiday'}
{'count': '1', 'name': 'play'}
{'count': '1', 'name': 'discworld'}
{'count': '1', 'name': 'reference'}
{'count': '2', 'name': 'science-fantasy'}
{'count': '1', 'name': 'school'}
{'count': '3', 'name': 'classics'}
{'count': '1', 'name': 'science-fiction'}


In [78]:
df.shape

(15714, 18)

In [79]:
df.head()

,work_id,clean_title,authors,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,popular_shelves,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,author_ids,num_pages_missing,popular_shelves_cleaned
0,166,Laocoon: An Essay on the Limits of Painting an...,"[{'author_id': '7723', 'role': ''}, {'author_i...",[],[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],"Originally published in 1766, the Laocoonhas b...","[{'count': '76', 'name': 'to-read'}, {'count':...","[814670, 30529, 117249, 2973210, 609042, 94903...",9,319,3.93,296.0,"[7723, 3855172]",0,"[{'count': '59', 'name': 'art'}, {'count': '30..."
1,466,The Melancholy Death of Oyster Boy & Other Sto...,"[{'author_id': '5773', 'role': ''}]",[],"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...",Occupying a similarly sinister and macabre wor...,"[{'count': '12411', 'name': 'to-read'}, {'coun...","[205981, 193264, 36024, 47561, 223851, 58096, ...",999,20556,4.13,128.0,[5773],0,"[{'count': '881', 'name': 'poetry'}, {'count':..."
2,527,Azul,"[{'author_id': '25575', 'role': ''}]",[],"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","La publicacion de 'Azul', primer libro de un e...","[{'count': '792', 'name': 'to-read'}, {'count'...","[1073063, 167425, 1375835, 144611, 214319, 953...",1,10,3.88,140.0,[25575],0,"[{'count': '45', 'name': 'poetry'}, {'count': ..."
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[{'author_id': '947', 'role': ''}, {'author_id...",[],[0451530357],[9780451530356],[104870],[Signet],The plays collected here follow: the journeys ...,"[{'count': '41', 'name': 'to-read'}, {'count':...",[],4,31,3.97,736.0,"[947, 30166]",0,"[{'count': '14', 'name': 'play'}, {'count': '7..."
4,1140,100 Selected Poems,"[{'author_id': '10547', 'role': ''}]",[],[0802130720],[9780802130723],[76889],[Grove Press],E.E. Cummings is without question one of the m...,"[{'count': '34800', 'name': 'to-read'}, {'coun...","[75504, 367802, 65336, 47733, 133380, 46201, 4...",369,22739,4.32,128.0,[10547],0,"[{'count': '1884', 'name': 'poetry'}, {'count'..."


In [80]:
df.columns

Index(['work_id', 'clean_title', 'authors', 'series_list', 'isbn_list',
       'isbn13_list', 'book_id_list', 'publisher_list', 'description',
       'popular_shelves', 'similar_books_list', 'text_reviews_count',
       'ratings_count', 'average_rating', 'num_pages', 'author_ids',
       'num_pages_missing', 'popular_shelves_cleaned'],
      dtype='str')

In [81]:
df = df[['work_id', 'clean_title', 'author_ids', 'series_list', 'isbn_list', 
         'isbn13_list', 'book_id_list', 'publisher_list', 'description', 
         'similar_books_list', 'text_reviews_count', 'ratings_count', 
         'average_rating', 'num_pages', 'num_pages_missing', 'popular_shelves_cleaned']]

In [82]:
df.head()

,work_id,clean_title,author_ids,series_list,isbn_list,isbn13_list,book_id_list,publisher_list,description,similar_books_list,text_reviews_count,ratings_count,average_rating,num_pages,num_pages_missing,popular_shelves_cleaned
0,166,Laocoon: An Essay on the Limits of Painting an...,"[7723, 3855172]",[],[0801831393],[9780801831393],[12158],[Johns Hopkins University Press],"Originally published in 1766, the Laocoonhas b...","[814670, 30529, 117249, 2973210, 609042, 94903...",9,319,3.93,296.0,0,"[{'count': '59', 'name': 'art'}, {'count': '30..."
1,466,The Melancholy Death of Oyster Boy & Other Sto...,[5773],[],"[057122444X, 0571270247, 0571195121, 068815681...","[9780571224449, 9780571270248, 9780571195121, ...","[945817, 8713848, 18743, 519112, 18742]","[Faber and Faber, Faber & Faber, Faber, Rob We...",Occupying a similarly sinister and macabre wor...,"[205981, 193264, 36024, 47561, 223851, 58096, ...",999,20556,4.13,128.0,0,"[{'count': '881', 'name': 'poetry'}, {'count':..."
2,527,Azul,[25575],[],"[1607962098, 9687748370]","[9781607962090, 9789687748375]","[7769271, 7490229]","[www.bnpublishing.com, LD Books]","La publicacion de 'Azul', primer libro de un e...","[1073063, 167425, 1375835, 144611, 214319, 953...",1,10,3.88,140.0,0,"[{'count': '45', 'name': 'poetry'}, {'count': ..."
3,723,Pericles/Cymbeline/The Two Noble Kinsmen,"[947, 30166]",[],[0451530357],[9780451530356],[104870],[Signet],The plays collected here follow: the journeys ...,[],4,31,3.97,736.0,0,"[{'count': '14', 'name': 'play'}, {'count': '7..."
4,1140,100 Selected Poems,[10547],[],[0802130720],[9780802130723],[76889],[Grove Press],E.E. Cummings is without question one of the m...,"[75504, 367802, 65336, 47733, 133380, 46201, 4...",369,22739,4.32,128.0,0,"[{'count': '1884', 'name': 'poetry'}, {'count'..."


In [83]:
df.shape

(15714, 16)

In [84]:
df.to_csv('../data/intermediate/books-cleaned/poetry.csv', index=False)